# AgentMesh on Colab

One notebook for everything: set up, check the code, run evals, inspect results, open the dashboard.
Run **Section 1** every session, then jump to the section you need.

| Section | Runtime | When |
|---|---|---|
| 1. Setup | any | every session |
| 2. Checks | CPU is enough | after pulling new code |
| 3. Evals | **T4 GPU** | to measure a change (~55 min for all 50 tasks) |
| 4. Results | CPU is enough | to regrade, compare and read answers |
| 5. Dashboard | CPU is enough | to browse runs and trajectories in the UI |
| 6. Troubleshooting | varies | only if something breaks |

Everything durable lives in Drive under `MyDrive/agentmesh_runs/`, so a disconnect never loses a run.

## 1. Setup

In [ ]:
# Drive for durable results; models cache on local disk (a Drive cache corrupted weights before)
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = "/content/hf_cache"
os.environ["EVAL_MIRROR_DIR"] = "/content/drive/MyDrive/agentmesh_runs"  # every results file is copied here
os.makedirs(os.environ["EVAL_MIRROR_DIR"], exist_ok=True)

In [ ]:
# Clone once, then update in place. Never `rm -rf` the repo: data/ holds this session's runs.
# BRANCH = "main" normally; set it to a PR branch (e.g. "pr/16-dashboard-correctness") to test before merging.
BRANCH = "main"

import os
if not os.path.exists("/content/agentmesh"):
    !git clone https://github.com/jyotishmann/agentmesh.git /content/agentmesh
%cd /content/agentmesh
!git fetch origin {BRANCH} && git checkout -B {BRANCH} origin/{BRANCH}
!git log --oneline -1

In [ ]:
# Dependencies and the runtime folders the app writes to
!pip install -r requirements.txt -q
!mkdir -p data/sandbox data/knowledge_base data/eval_results

## 2. Checks (CPU is enough)

The pre-flight scan catches calls to methods that don't exist and `settings.<field>` names missing
from `config.py`, without loading a model. The tests cover the API routes, the grader and the
eval reliability code: **expect 102 passed**.

`tests/test_config.py`, `test_memory.py` and `test_tools.py` are left out on purpose: 16 of their
tests were written against older constructors and fail on `main` too. They're fixed in a follow-up PR.

In [ ]:
# Pre-flight: missing methods and bad settings fields, found by reading the code (no models loaded)
import ast, pathlib
from agentmesh.config import Settings

root = pathlib.Path("/content/agentmesh/agentmesh")
valid_settings = set(Settings.model_fields)
classes, problems = {}, []

for py in root.rglob("*.py"):
    tree = ast.parse(py.read_text())
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            defined = {n.name for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))}
            bases = [b.id for b in node.bases if isinstance(b, ast.Name)]
            classes[node.name] = (defined, bases, node, py)
        if isinstance(node, ast.Attribute) and isinstance(node.value, ast.Name) \
                and node.value.id == "settings" and node.attr not in valid_settings:
            problems.append(f"{py.relative_to(root)}:{node.lineno}  settings.{node.attr} not in config.py")

def all_methods(name, seen=None):
    seen = seen or set()
    if name not in classes or name in seen:
        return set()
    seen.add(name)
    defined, bases, _, _ = classes[name]
    return defined | set().union(*(all_methods(b, seen) for b in bases)) if bases else defined

for name, (_, _, node, py) in classes.items():
    # Attributes set as self.x = ... are callable objects (e.g. a tokenizer), not missing methods
    attributes = {t.attr for sub in ast.walk(node) if isinstance(sub, ast.Assign) for t in sub.targets
                  if isinstance(t, ast.Attribute) and isinstance(t.value, ast.Name) and t.value.id == "self"}
    methods = all_methods(name) | attributes
    for sub in ast.walk(node):
        if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) \
                and isinstance(sub.func.value, ast.Name) and sub.func.value.id == "self" \
                and sub.func.attr.startswith("_") and sub.func.attr not in methods:
            problems.append(f"{py.relative_to(root)}:{sub.lineno}  {name}.{sub.func.attr}() not defined")

print("\n".join(problems) if problems else "✅ No missing methods or bad settings fields found")

In [ ]:
# GPU-free tests (expect 102 passed)
!python -m pytest tests/test_api.py tests/test_api_routes.py tests/test_correctness.py tests/test_reliability.py tests/test_grader.py -q

## 3. Evals (T4 GPU)

Runs are deterministic (greedy decoding) and memory-isolated, save after every task, and mirror to Drive.
If Colab disconnects, reconnect, run Section 1, then use the resume cell.

In [ ]:
# Confirm the GPU and how much memory is free (expect ~15 GB on a T4)
import torch
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU for this section"
free, total = torch.cuda.mem_get_info()
print(f"{torch.cuda.get_device_name(0)}: {free / 1e9:.1f} / {total / 1e9:.1f} GB free")

In [ ]:
# Full suite: 50 tasks (~55 min). For a quick check, add --max-difficulty 1 (~5 min)
!python -m agentmesh.eval.cli run

In [ ]:
# One line per saved run, local and on Drive. Interrupted runs are marked "incomplete".
!python -m agentmesh.eval.cli history

In [ ]:
# Resume an interrupted run: only unfinished tasks run, with the original filters.
# Paste the file name from the history above, then uncomment.
# !python -m agentmesh.eval.cli run --resume /content/drive/MyDrive/agentmesh_runs/eval_YYYYMMDD_HHMMSS.json

In [ ]:
# Back up the trajectories: regrading a run later needs the answers stored here
!cp data/trajectories.db "/content/drive/MyDrive/agentmesh_runs/trajectories_$(date +%Y%m%d_%H%M%S).db"
!ls -la /content/drive/MyDrive/agentmesh_runs/

## 4. Results (CPU is enough)

Start here in a fresh session: the first cell copies saved runs from Drive into `data/`,
where the CLI and the dashboard look for them.

In [ ]:
# Restore saved runs from Drive (and the repo's results/ folder, if present) into data/
import glob, os, shutil

runs_dir = os.environ["EVAL_MIRROR_DIR"]
sources = glob.glob(f"{runs_dir}/eval_*.json") + glob.glob("results/eval_*.json")
copied = 0
for src in sources:
    dst = f"data/eval_results/{os.path.basename(src)}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        copied += 1
print(f"Copied {copied} results files; data/eval_results has {len(glob.glob('data/eval_results/eval_*.json'))}")

# Keep this session's trajectories if there are any; otherwise use the newest backup
backups = sorted(glob.glob(f"{runs_dir}/trajectories_*.db") + glob.glob("results/trajectories*.db"))
if not os.path.exists("data/trajectories.db") and backups:
    shutil.copy(backups[-1], "data/trajectories.db")
    print(f"Trajectories restored from {os.path.basename(backups[-1])}")
else:
    print("Trajectories:", "this session's data/trajectories.db" if os.path.exists("data/trajectories.db") else "none found")

In [ ]:
# Re-score the newest run with the current grader (no GPU). Writes <name>_regraded.json, mirrored to Drive.
# The trajectories database must be the one saved with that run (Section 3's backup), or tasks are skipped.
import glob
runs = sorted(p for p in glob.glob("data/eval_results/eval_*.json") if not p.endswith("_regraded.json"))
latest = runs[-1]
print("Regrading", latest)
!python -m agentmesh.eval.cli regrade {latest} --db data/trajectories.db

In [ ]:
# Every saved run side by side, correctness first
import glob, json, os

def pct(v):
    return "  n/a" if v is None else f"{v * 100:4.0f}%"

print(f"{'run':<34} {'correct':>7} {'critic agree':>12} {'false-pass':>10} {'complete':>8}")
for path in sorted(glob.glob("data/eval_results/eval_*.json")):
    run = json.load(open(path))
    o = run["summary"]["overall"]
    flag = "" if run.get("finished", True) else "  (incomplete)"
    print(f"{os.path.basename(path):<34} {pct(o.get('answer_correct')):>7} {pct(o.get('critic_agreement')):>12} "
          f"{pct(o.get('critic_false_pass_rate')):>10} {pct(o.get('task_completion')):>8}{flag}")

In [ ]:
# Every wrong answer in a run, next to what was expected and what the critic said
import glob, json, sqlite3

RUN = sorted(glob.glob("data/eval_results/eval_*.json"))[-1]  # or paste a path
run = json.load(open(RUN))
tasks = {t["task_id"]: t for t in json.load(open("agentmesh/eval/tasks.json"))}
con = sqlite3.connect("data/trajectories.db")

print(RUN, "\n")
for r in run["task_results"]:
    if r["metrics"].get("answer_correct") != 0.0:
        continue
    row = con.execute("SELECT final_output FROM trajectories WHERE task_id = ?", (r["trajectory_id"],)).fetchone()
    answer = (row[0] if row else "(trajectory not in this database)").strip().replace("\n", " ")
    critic = {1.0: "PASSED it", 0.0: "rejected it"}.get(r["metrics"].get("critic_pass"), "-")
    print(f"=== {r['task_id']} | critic {critic}")
    print(f"  EXPECTED: {tasks[r['task_id']]['expected']}")
    print(f"  GOT:      {answer[:400]}\n")

## 5. Dashboard (CPU is enough)

Browsing runs and trajectories needs no GPU: the Qwen models load only when a chat or eval starts.
The first start downloads the small embedding model (~130 MB). Run Section 4's first cell before this,
so there's something to show.

In [ ]:
# Start the API server in the background and wait until it answers
import subprocess, time, requests

api_process = subprocess.Popen(
    ["uvicorn", "agentmesh.api.server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("api.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(90):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).ok:
            print("API up.")
            break
    except requests.ConnectionError:
        time.sleep(2)
else:
    print("API did not start. Last lines of api.log:")
    !tail -20 api.log

In [ ]:
# Start Streamlit and a public tunnel to it. Open the URL; the page asks for the password printed here.
!npm install -g localtunnel -q

ui_process = subprocess.Popen(
    ["streamlit", "run", "agentmesh/ui/app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=open("ui.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(5)
print("Tunnel password:", requests.get("https://loca.lt/mytunnelpassword", timeout=10).text.strip())
tunnel_process = subprocess.Popen(["npx", "localtunnel", "--port", "8501"], stdout=subprocess.PIPE, text=True)
print(tunnel_process.stdout.readline().strip())  # "your url is: https://....loca.lt"

In [ ]:
# Stop the dashboard
for p in ("tunnel_process", "ui_process", "api_process"):
    if p in globals():
        globals()[p].terminate()
print("Stopped.")

## 6. Troubleshooting

In [ ]:
# SafetensorError or "incomplete metadata"? Check every cached weights file and delete damaged models
import glob, json, os, shutil, struct

roots = {os.environ.get("HF_HOME"), os.path.expanduser("~/.cache/huggingface"), "/content/drive/MyDrive/hf_cache"}
roots = sorted(r for r in roots if r and os.path.isdir(r))

def check(path):
    """None if the file is intact, else why it's damaged."""
    size = os.path.getsize(path)
    if size < 8:
        return f"only {size} bytes"
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if header_len > size - 8:
            return "impossible header length"
        try:
            header = json.loads(f.read(header_len))
        except ValueError:
            return "header is not valid JSON"
    data_end = max((t["data_offsets"][1] for k, t in header.items() if k != "__metadata__"), default=0)
    expected = 8 + header_len + data_end
    return None if size == expected else f"truncated: {size / 1e6:.1f} of {expected / 1e6:.1f} MB"

damaged = set()
for path in sorted({p for r in roots for p in glob.glob(f"{r}/**/*.safetensors", recursive=True)}):
    problem = check(path)
    print(("OK      " if problem is None else "DAMAGED ") + path + (f"  <- {problem}" if problem else ""))
    if problem:
        parts = path.split(os.sep)
        damaged.add(next((os.sep.join(parts[:i + 1]) for i, p in enumerate(parts) if p.startswith("models--")),
                         os.path.dirname(path)))
for model_dir in sorted(damaged):
    shutil.rmtree(model_dir, ignore_errors=True)
    print("Deleted", model_dir, "- it re-downloads on next use")

In [ ]:
# End-to-end smoke test on the GPU: one task through planner, specialists and critic
from agentmesh.orchestrator import Orchestrator

orch = Orchestrator()
result = orch.run("What is 2 + 2?")
print(f"Response: {result.response}\nCompleted: {result.completed}")
for e in result.trajectory_events:
    print(f"  {e['step_number']:>2}. {e['agent_name']:<14} {e['action_type']:<14} {e.get('tool_name', '')}")